# Broadcasting in Neural Networks

**Goal:** Show how broadcasting powers core neural-network operations — batched linear layers, per-feature affine scale/shift (batch-norm style), and attention-style bias addition — implementing each from scratch in PyTorch, validating against `nn.Linear` and `torch.autograd`, then replacing with idiomatic equivalents.

Cross-links: `[[broadcasting]]` (general tensor rule), `[[matrix-multiplication]]` (affine layers), `[[backpropagation]]` (summed gradients for broadcasted parameters), `[[batch-normalization]]` (scale/shift).

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Broadcasting Rules Recap

Two dimensions are **broadcast-compatible** when they are equal or one of them is `1`.  
Alignment starts from the **trailing** dimension and works left.

Key rule for backpropagation: if a tensor of shape `(1, D)` is broadcast to `(B, D)` in the forward pass,  
the reverse-mode gradient must **sum over every axis that was expanded** — reducing `(B, D)` back to `(1, D)`.

| Forward broadcast | Gradient reduction |
|---|---|
| `(D,) → (B, D)` | `sum over dim 0` |
| `(1, D) → (B, D)` | `sum over dim 0, keepdim=True` |
| `(C, 1, 1) → (B, C, H, W)` | `sum over dims 0, 2, 3` |

## Part 1 — Batched Linear Layer: `X @ W.T + b`

`nn.Linear(in, out)` computes `y = X @ W.T + b` where  
- `X` has shape `(B, Din)` — a mini-batch of `B` examples  
- `W` has shape `(Dout, Din)` — each row is a weight vector for one output neuron  
- `b` has shape `(Dout,)` — one bias per output feature, **broadcast across the batch**

The bias broadcast: `(Dout,)` aligns with trailing dim of `(B, Dout)`, expanding to `(B, Dout)` without copying data.

In [2]:
import torch.nn as nn

torch.manual_seed(0)

B, Din, Dout = 8, 16, 32

# Random input batch
X = torch.randn(B, Din, device=device)

# From-scratch batched linear layer ------------------------------------------------
W_scratch = torch.randn(Dout, Din, device=device)   # shape (Dout, Din)
b_scratch = torch.randn(Dout, device=device)         # shape (Dout,)

# Matrix multiply: (B, Din) @ (Din, Dout) -> (B, Dout)
# b_scratch has shape (Dout,); broadcasting aligns it with trailing dim -> (B, Dout)
Z_scratch = X @ W_scratch.T + b_scratch             # b broadcasts over batch axis

print(f"X.shape         = {X.shape}")
print(f"W_scratch.shape = {W_scratch.shape}")
print(f"b_scratch.shape = {b_scratch.shape}")
print(f"Z_scratch.shape = {Z_scratch.shape}  <- (B, Dout) as expected")

# Confirm b did NOT replicate into a (B, Dout) tensor — still shape (Dout,)
print(f"b_scratch still has shape {b_scratch.shape} (no copy)")

X.shape         = torch.Size([8, 16])
W_scratch.shape = torch.Size([32, 16])
b_scratch.shape = torch.Size([32])
Z_scratch.shape = torch.Size([8, 32])  <- (B, Dout) as expected
b_scratch still has shape torch.Size([32]) (no copy)


### Validation: Compare with `nn.Linear`

We copy our weight/bias into an `nn.Linear` module and verify the outputs are identical.

In [3]:
# Build nn.Linear and copy our weights into it
linear = nn.Linear(Din, Dout, bias=True).to(device)
with torch.no_grad():
    linear.weight.copy_(W_scratch)   # nn.Linear stores W as (Dout, Din)
    linear.bias.copy_(b_scratch)

Z_torch = linear(X)

assert torch.allclose(Z_scratch, Z_torch, atol=1e-5), (
    f"Max deviation: {(Z_scratch - Z_torch).abs().max().item():.2e}"
)
print("PASS: manual broadcast linear matches nn.Linear")
print(f"Max abs diff: {(Z_scratch - Z_torch).abs().max().item():.2e}")

PASS: manual broadcast linear matches nn.Linear
Max abs diff: 0.00e+00


### Gradient Check: Bias Gradient = Sum Over Batch

The bias `b` participates in `B` independent forward computations (once per example).  
By the chain rule, `dL/db = sum_{i=1}^{B} dL/dZ_i` — the batch dimension is summed.

We verify this against `torch.autograd`.

In [4]:
# Re-create with requires_grad so autograd can track
W_ag = W_scratch.detach().clone().requires_grad_(True)
b_ag = b_scratch.detach().clone().requires_grad_(True)

Z_ag = X @ W_ag.T + b_ag          # bias broadcasts: (Dout,) -> (B, Dout)
loss = Z_ag.sum()                  # scalar loss: dL/dZ = ones, shape (B, Dout)
loss.backward()

# Manual gradient for bias: sum upstream gradient over batch axis
upstream = torch.ones(B, Dout, device=device)  # dL/dZ
db_manual = upstream.sum(dim=0)                 # shape (Dout,)

assert torch.allclose(b_ag.grad, db_manual, atol=1e-5), (
    f"Bias grad mismatch: max diff = {(b_ag.grad - db_manual).abs().max().item():.2e}"
)
print("PASS: bias gradient is sum of upstream over batch axis")
print(f"b_ag.grad.shape = {b_ag.grad.shape}  (Dout,)")
print(f"db_manual.shape = {db_manual.shape}  (Dout,)")
print(f"Max abs diff:     {(b_ag.grad - db_manual).abs().max().item():.2e}")

PASS: bias gradient is sum of upstream over batch axis
b_ag.grad.shape = torch.Size([32])  (Dout,)
db_manual.shape = torch.Size([32])  (Dout,)
Max abs diff:     0.00e+00


## Part 2 — Per-Feature Affine Scale / Shift (Batch-Norm Style)

Batch normalization learns a `gamma` (scale) and `beta` (shift) per **feature dimension**.  
For an activation batch `A` of shape `(B, D)`:

```
out = gamma * A + beta
```

where `gamma` and `beta` have shape `(D,)` and broadcast over the batch axis.

In [5]:
torch.manual_seed(1)

B, D = 12, 64

A = torch.randn(B, D, device=device)
gamma = torch.randn(D, device=device)   # per-feature scale
beta  = torch.randn(D, device=device)   # per-feature shift

# From-scratch: gamma and beta broadcast over axis 0 (batch)
out_scratch = gamma * A + beta          # both (D,) broadcast with (B, D) -> (B, D)

print(f"A.shape      = {A.shape}")
print(f"gamma.shape  = {gamma.shape}")
print(f"out.shape    = {out_scratch.shape}")

# Idiomatic: identical — PyTorch broadcasting already handles (D,) * (B, D)
out_torch = gamma * A + beta            # same expression; shown for clarity
assert torch.allclose(out_scratch, out_torch), "scale/shift mismatch"
print("PASS: per-feature affine matches expected (B, D) output")

A.shape      = torch.Size([12, 64])
gamma.shape  = torch.Size([64])
out.shape    = torch.Size([12, 64])
PASS: per-feature affine matches expected (B, D) output


In [6]:
# Gradient check for gamma: dL/dgamma = sum over batch
gamma_ag = gamma.detach().clone().requires_grad_(True)
beta_ag  = beta.detach().clone().requires_grad_(True)

out_ag = gamma_ag * A + beta_ag
loss_ag = out_ag.sum()
loss_ag.backward()

# Manual: upstream is ones(B, D); dgamma = sum over dim 0 (batch)
upstream = torch.ones(B, D, device=device)
dgamma_manual = (upstream * A).sum(dim=0)   # chain rule: d(gamma * A)/d(gamma) = A
dbeta_manual  = upstream.sum(dim=0)

assert torch.allclose(gamma_ag.grad, dgamma_manual, atol=1e-5), "gamma grad mismatch"
assert torch.allclose(beta_ag.grad,  dbeta_manual,  atol=1e-5), "beta grad mismatch"
print("PASS: gamma and beta gradients match autograd")
print(f"gamma.grad.shape = {gamma_ag.grad.shape}")
print(f"beta.grad.shape  = {beta_ag.grad.shape}")

PASS: gamma and beta gradients match autograd
gamma.grad.shape = torch.Size([64])
beta.grad.shape  = torch.Size([64])


## Part 3 — Attention-Style Bias: `(B, 1, D) + (1, T, D)`

In attention mechanisms it is common to combine two tensors with complementary singleton dimensions,  
letting them broadcast into a shared `(B, T, D)` shape without materialising any copy.

Example: a query context vector of shape `(B, 1, D)` and a key-dependent bias of shape `(1, T, D)`  
combine to produce a `(B, T, D)` result — each `(b, t)` pair gets `context[b] + key_bias[t]`.

In [7]:
torch.manual_seed(2)

B, T, D_attn = 4, 10, 8

context    = torch.randn(B, 1, D_attn, device=device)   # (B, 1, D)
key_bias   = torch.randn(1, T, D_attn, device=device)   # (1, T, D)

# Natural broadcast
out_broadcast = context + key_bias                       # (B, T, D) via broadcasting

print(f"context.shape   = {context.shape}")
print(f"key_bias.shape  = {key_bias.shape}")
print(f"out.shape       = {out_broadcast.shape}  <- (B, T, D)")

# Manual broadcast: expand both to (B, T, D) then add
context_exp  = context.expand(B, T, D_attn)
key_bias_exp = key_bias.expand(B, T, D_attn)
out_manual   = context_exp + key_bias_exp

assert torch.allclose(out_broadcast, out_manual), "attention bias mismatch"
print("PASS: natural broadcast equals manually expanded result")

# Confirm expand uses strides (no copy): same storage as original
print(f"context_exp.is_contiguous() = {context_exp.is_contiguous()}  (False means stride-0 view)")

context.shape   = torch.Size([4, 1, 8])
key_bias.shape  = torch.Size([1, 10, 8])
out.shape       = torch.Size([4, 10, 8])  <- (B, T, D)


PASS: natural broadcast equals manually expanded result
context_exp.is_contiguous() = False  (False means stride-0 view)


In [8]:
# Gradient check for (B, 1, D) + (1, T, D)
context_ag  = context.detach().clone().requires_grad_(True)
key_bias_ag = key_bias.detach().clone().requires_grad_(True)

out_ag2 = context_ag + key_bias_ag        # (B, T, D)
loss_ag2 = out_ag2.sum()
loss_ag2.backward()

# Manual: dL/d_context sums over T (was broadcast); dL/d_key_bias sums over B
upstream2 = torch.ones(B, T, D_attn, device=device)
d_context_manual  = upstream2.sum(dim=1, keepdim=True)   # (B, 1, D)
d_key_bias_manual = upstream2.sum(dim=0, keepdim=True)   # (1, T, D)

assert torch.allclose(context_ag.grad,  d_context_manual,  atol=1e-5), "context grad mismatch"
assert torch.allclose(key_bias_ag.grad, d_key_bias_manual, atol=1e-5), "key_bias grad mismatch"
print("PASS: attention bias gradients match autograd")
print(f"context.grad.shape  = {context_ag.grad.shape}   <- (B, 1, D), summed over T")
print(f"key_bias.grad.shape = {key_bias_ag.grad.shape}  <- (1, T, D), summed over B")

PASS: attention bias gradients match autograd
context.grad.shape  = torch.Size([4, 1, 8])   <- (B, 1, D), summed over T
key_bias.grad.shape = torch.Size([1, 10, 8])  <- (1, T, D), summed over B


## Idiomatic PyTorch

All three patterns above are already first-class PyTorch:

| Operation | Idiomatic form |
|---|---|
| Batched linear + bias | `nn.Linear(in, out)` |
| Per-feature scale/shift | `gamma * x + beta` (broadcast automatic) |
| Attention-style bias | `q + k` where shapes have complementary singletons |

PyTorch uses **stride-0 views** for broadcasting — no data is copied unless you call `.contiguous()`.  
Use `tensor.expand(...)` to create an explicit view, or `tensor.repeat(...)` to materialise a copy  
(needed when the consumer requires contiguous memory, e.g. some CUDA kernels).

In [9]:
# Idiomatic linear (already validated above)
linear_idem = nn.Linear(Din, Dout).to(device)
with torch.no_grad():
    linear_idem.weight.copy_(W_scratch)
    linear_idem.bias.copy_(b_scratch)
z_idem = linear_idem(X)
assert torch.allclose(z_idem, Z_scratch, atol=1e-5)
print("PASS: nn.Linear (idiomatic) matches from-scratch result")

# Idiomatic per-feature affine (same expression, confirming it's already idiomatic)
out_idem = gamma * A + beta
assert torch.allclose(out_idem, out_scratch, atol=1e-5)
print("PASS: gamma*A+beta (idiomatic) matches from-scratch result")

# Idiomatic attention-style
out_attn_idem = context + key_bias
assert torch.allclose(out_attn_idem, out_broadcast, atol=1e-5)
print("PASS: context + key_bias (idiomatic) matches from-scratch result")

PASS: nn.Linear (idiomatic) matches from-scratch result
PASS: gamma*A+beta (idiomatic) matches from-scratch result
PASS: context + key_bias (idiomatic) matches from-scratch result


## Visualisation — Bias Gradient Accumulation

The plot below shows bias gradient magnitude per feature (summed over a batch of 32),  
illustrating that each feature's gradient accumulates contributions from all `B` examples.

In [10]:
torch.manual_seed(3)
B_vis, D_vis = 32, 16

X_vis  = torch.randn(B_vis, D_vis, device=device)
W_vis  = torch.randn(D_vis, D_vis, device=device)
b_vis  = torch.randn(D_vis, device=device, requires_grad=True)

Z_vis = X_vis @ W_vis.T + b_vis
loss_vis = (Z_vis ** 2).sum()
loss_vis.backward()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(D_vis), b_vis.grad.detach().cpu().numpy(), color="steelblue")
axes[0].set_title(f"Bias gradient (B={B_vis})")
axes[0].set_xlabel("Feature index")
axes[0].set_ylabel("dL/db")

# Show first 4 per-example contributions (upstream * 1)
sample_contribs = (2 * Z_vis[:4]).detach().cpu().numpy()  # dL/dZ = 2Z for L=Z^2
for i, row in enumerate(sample_contribs):
    axes[1].plot(row, alpha=0.7, label=f"example {i}")
axes[1].set_title("Per-example upstream gradients (first 4)")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("dL/dZ")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("broadcasting_gradients.png", dpi=80)
plt.close()
print("Plot saved to broadcasting_gradients.png")

Plot saved to broadcasting_gradients.png


## Takeaways

1. **Broadcasting is parameter sharing.** When `b` of shape `(D,)` is added to `X` of shape `(B, D)`,  
   the same bias value is used for all `B` examples in the batch — a form of weight sharing.

2. **Sharing in the forward pass → summation in the backward pass.**  
   Gradients for broadcasted axes must be summed back to the original shape:  
   `dL/db = sum over batch of dL/dZ`.

3. **Shape alignment is right-to-left (trailing dims first).**  
   A tensor of shape `(C,)` aligns by default to the **last** axis (width) of `(B, C, H, W)`, which is NOT the channel axis;  
   to apply a per-channel op you must reshape to `(C, 1, 1)` so it aligns to the channel axis and broadcasts across `B`, `H`, `W`.

4. **Broadcasting does not copy data.**  
   `expand()` returns a stride-0 view; actual duplication only happens with `repeat()` or `.contiguous()`.

5. **Silent shape bugs are the main risk.**  
   `(B, 1) - (B,)` expands to `(B, B)` rather than `(B,)` — always assert output shapes in new code.

Cross-links: `[[broadcasting]]` — `[[backpropagation]]` — `[[batch-normalization]]`